# Build the EN tool-calling audio dataset — Voxtral TTS -> HF (L4)

La generation texte (Gemini) se fait **en local sur ton Mac** (aucun GPU). Ce notebook
**L4** ne fait que : recuperer le JSONL texte -> synthese voix **Voxtral TTS** (vLLM-Omni,
en parallele) -> assemblage `DatasetDict` -> push HF (prive).

**Temps attendu (3000 exemples)** : telechargement+chargement Voxtral ~5-10 min,
puis TTS ~5-15 min a `--concurrency 8` (vLLM batch les requetes) -> **~15-25 min** au total.
Le script affiche un **ETA en direct** des les 50 premieres requetes.

**Licence** : audio synthetise par un modele CC-BY-NC-4.0 -> dataset non-commercial, garde-le prive.


## 1. Installer les dependances
vLLM et vLLM-Omni doivent etre **apparies** (memes major.minor) : vllm-omni ne declare
pas vllm en dependance, et `-U` casse l'appariement (vllm 0.23 vs vllm-omni 0.22 ->
`ModuleNotFoundError: vllm.entrypoints.utils`). On epingle donc la paire 0.22.0.

In [ ]:
import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("vllm==0.22.0", "vllm-omni==0.22.0")             # paire APPARIEE (Voxtral TTS)
pip("httpx", "soundfile", "torchaudio", "datasets", "huggingface_hub", "mistral_common")
print("deps ok")

## 2. Recuperer le repo

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Rcarvalo/finetuning_s2s_toolcalling"   # <-- ton repo
BRANCH   = "claude/blissful-tesla-7i1yky"                              # <-- ta branche
WORK     = "/content/finetuning_s2s_toolcalling"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK + "/src"); sys.path.insert(0, WORK + "/scripts")
print("repo:", WORK)

## 3. Cles + recuperer le JSONL texte (genere en local)
Le fichier `data/tc_en_train.jsonl` (~2 Mo) a ete genere sur ton Mac. Deux options :

- **A. Upload HF (reproductible)** — depuis le Mac : `huggingface_hub.upload_file(... repo_type='dataset')`
  vers un repo prive (ex. `Rcarvalo/tc-en-src`), puis la cellule ci-dessous le telecharge.
- **B. Drag-and-drop** — glisse `tc_en_train.jsonl` dans le panneau Fichiers de Colab,
  dans `/content/finetuning_s2s_toolcalling/data/`, et **saute** le telechargement.

In [ ]:
from huggingface_hub import login, hf_hub_download
login()                                          # token HF (write)
HF_REPO = "Rcarvalo/tc-en-audio-toolcalling"     # <-- ton dataset final (sera prive)

SRC_REPO = "Rcarvalo/tc-en-src"                  # <-- repo ou tu as uploade le JSONL (option A)
import os, shutil
DST = "data/tc_en_train.jsonl"
if not os.path.exists(DST):
    p = hf_hub_download(SRC_REPO, "tc_en_train.jsonl", repo_type="dataset")
    os.makedirs("data", exist_ok=True); shutil.copy(p, DST)
print("dialogues:", subprocess.run(["wc", "-l", DST], capture_output=True, text=True).stdout.strip())

## 4. Lancer Voxtral TTS (vLLM-Omni) en arriere-plan
Le wheel vLLM est builde **CUDA 13** -> on pointe `LD_LIBRARY_PATH` vers `libcudart.so.13`
(sinon `ImportError: libcudart.so.13`). Si le serveur meurt, les logs s'affichent tout de suite.

In [ ]:
import os, sys, glob, subprocess, time, httpx
libs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libcudart.so.13", recursive=True)
if not libs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu13"], check=False)
    libs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libcudart.so.13", recursive=True)
libdirs = sorted({os.path.dirname(p) for p in libs})
env = dict(os.environ); env["LD_LIBRARY_PATH"] = ":".join(libdirs + [env.get("LD_LIBRARY_PATH", "")])
print("cu13 libdirs:", libdirs or "INTROUVABLE")

LOG = open("/content/voxtral.log", "w")
srv = subprocess.Popen(["vllm", "serve", "mistralai/Voxtral-4B-TTS-2603", "--omni"],
                       env=env, stdout=LOG, stderr=subprocess.STDOUT)
for _ in range(360):                            # download + load (~5-10 min)
    if srv.poll() is not None:
        print("SERVEUR MORT, derniers logs:\n", open("/content/voxtral.log").read()[-2500:]); break
    try:
        if httpx.get("http://localhost:8000/v1/models", timeout=5).status_code == 200:
            print("Voxtral pret."); break
    except Exception:
        pass
    time.sleep(10)

## 5. Choisir les voix
Voxtral expose 20 voix (visibles dans le log de demarrage : « Available voice embeddings »).
**5 sont de l'anglais natif** : `casual_male/female`, `cheerful_female`, `neutral_male/female`.
On entraine sur 3 et on garde 2 voix INCONNUES pour le test (vraie generalisation).
Les voix `fr_*/de_*/es_*/...` donnent de l'anglais ACCENTUE (robustesse) mais ecoute-les
d'abord (cellule 7) : la prononciation EN peut varier.

In [ ]:
TRAIN_VOICES = ["casual_male", "casual_female", "cheerful_female"]
TEST_VOICES  = ["neutral_male", "neutral_female"]   # held-out (absentes du train)

import httpx                         # probe : une voix repond bien ?
r = httpx.post("http://localhost:8000/v1/audio/speech", timeout=120,
               json={"input": "Quick test.", "model": "mistralai/Voxtral-4B-TTS-2603",
                     "response_format": "wav", "voice": TRAIN_VOICES[0]})
print("probe:", r.status_code, "->", len(r.content), "octets")

## 6. Synthetiser l'audio (parallele, ETA en direct)

In [ ]:
import sys, subprocess
# TRAIN (3000) -- voix train
subprocess.run([sys.executable, "scripts/synthesize_user_audio.py", "--engine", "voxtral",
    "--dialogues", "data/tc_en_train.jsonl", "--audio-root", "data/audio_tc_en",
    "--out", "data/tc_en_train.audio.jsonl", "--split", "train",
    "--concurrency", "8", "--voices", ",".join(TRAIN_VOICES)], check=True)
# BENCHMARK held-out -- voix de test
subprocess.run([sys.executable, "scripts/synthesize_user_audio.py", "--engine", "voxtral",
    "--dialogues", "benchmark/toolcalling_en/cases.sample.jsonl", "--audio-root", "data/audio_tc_en",
    "--out", "data/tc_en_bench.audio.jsonl", "--split", "test",
    "--concurrency", "8", "--voices", ",".join(TEST_VOICES)], check=True)

## 7. (option) Ecouter quelques exemples pour valider la voix

In [ ]:
import json, random
from IPython.display import Audio, display
rows = [json.loads(l) for l in open("data/tc_en_train.audio.jsonl")]
for r in random.sample(rows, 4):
    u = r["turns"][0]
    print(f"[{r['meta']['target']}] {u['text']}  (voice={u.get('voice')})")
    display(Audio(f"data/audio_tc_en/{u['audio']}"))

## 8. Assembler + pousser le dataset sur ton HF (prive, carte neutre)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "scripts/build_hf_dataset.py", "--repo-id", HF_REPO,
    "--train", "data/tc_en_train.audio.jsonl", "--test", "data/tc_en_bench.audio.jsonl",
    "--audio-root", "data/audio_tc_en", "--private"], check=True)
print("Dataset pousse:", f"https://huggingface.co/datasets/{HF_REPO}")

## Suite
Dataset audio pret sur ton HF. Entrainement ensuite : `configs/phase_en_toolcalling.yaml`
(LoRA backbone, encodeur + tetes audio geles), puis eval audio via
`scripts/eval_audio_toolcalling.py` (README, section *Capacite tool calling vocal EN*).